In [2]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
model = "claude-sonnet-4-6"

In [27]:
def add_user_message(messages, content):
    userMessage = {"role": "user", "content": content}
    messages.append(userMessage)

def add_assistant_message(messages, content):
    assistantMessage = {"role": "assistant", "content": content}
    messages.append(assistantMessage)

def chat(messages,system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 300,
        "messages": messages,
    }
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [28]:
messages = []
add_user_message(messages, "How do I solve 5x+3=2 and find x?")

system = """
        You are a math tutor, guide them step by step to the problem. Do not give direct answer.
    """
answer = chat(messages)
answer

'## Solving for x\n\n**Starting equation:**\n5x + 3 = 2\n\n**Step 1: Subtract 3 from both sides**\n\n5x + 3 - 3 = 2 - 3\n\n5x = -1\n\n**Step 2: Divide both sides by 5**\n\n5x/5 = -1/5\n\n**x = -1/5** (or -0.2)\n\n---\n\n**✓ Check the answer** by plugging back in:\n\n5(-1/5) + 3 = -1 + 3 = **2** ✔'

In [29]:
messages = []
system = """
You are a python engineer who writes very concise code. Don't write any comments or explanations. Just write the code.
"""
add_user_message(messages, "Write a python function to calculate factorial of a number")
answer = chat(messages, system)
answer

'```python\ndef factorial(n):\n    if n < 0:\n        raise ValueError("Factorial is not defined for negative numbers")\n    if n == 0 or n == 1:\n        return 1\n    return n * factorial(n - 1)\n```'

Streaming

In [30]:
messages = []
add_user_message(messages, "Write a one sentence story about a dog and a cat.")
stream = client.messages.create(
    model=model,
    max_tokens=300,
    messages=messages,
    stream=True
)
for chunk in stream:
    print(chunk)


RawMessageStartEvent(message=Message(id='msg_01HwsLsQouJBsXnSVGGGnxz4', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=19, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a one sentence story about a dog and a cat:\n\nAfter years of chasing each other around the house, the old', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDel

In [31]:
messages = []
add_user_message(messages, "Write a one sentence story about a dog and a cat.")
with client.messages.stream(
    model=model,
    max_tokens=300,
    messages=messages,
) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="")

stream.get_final_message()

Here is a one sentence story about a dog and a cat:

The old dog and the stray cat, once bitter enemies, curled up together on the porch on a cold winter's night and became the most unlikely of friends.

ParsedMessage(id='msg_01LqWNxL8hR5PdnegDBbu6ZS', container=None, content=[ParsedTextBlock(citations=None, text="Here is a one sentence story about a dog and a cat:\n\nThe old dog and the stray cat, once bitter enemies, curled up together on the porch on a cold winter's night and became the most unlikely of friends.", type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=19, output_tokens=52, server_tool_use=None, service_tier='standard'))

Stop sequence

In [38]:
# messages = []
# add_user_message(messages, "Write a python function to calculate factorial of a number")
# add_assistant_message(messages, "```python\n")

# answer = chat(messages, stop_sequences=["\n```"])
messages = []
add_user_message(messages, "Write a python function to calculate factorial of a number")
add_assistant_message(messages, "```python\n")

response = client.messages.create(
    model=model,
    max_tokens=300,
    messages=messages,
    stop_sequences=["```"]
)

answer = response.content[0].text
print(response.stop_reason)   # "stop_sequence" if matched
print(response.stop_sequence) # "```" if matched
answer


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages: final assistant content cannot end with trailing whitespace'}, 'request_id': 'req_011CbK7vJ69JUTPjoNafGv2F'}